# 4. Perbandingan Model dan Transfer Lintas Wilayah

Notebook terakhir dari empat notebook **Gabungan**. Tiga pertanyaan:

1. **Seberapa berbeda performa model Bantul, Sleman, dan Gabungan?**
2. **Apakah variabel yang dianggap penting oleh model sama di kedua wilayah?**
3. **Apakah model yang dilatih di satu kabupaten masih berguna di kabupaten
   lain?** -- pertanyaan yang tidak bisa dijawab satu wilayah saja, dan yang
   paling menentukan apakah pendekatan ini layak dipakai di luar tempat
   datanya dikumpulkan.

Notebook ini **tidak melatih apa pun**. Ia membaca hasil yang sudah ada:
`ringkasan_model.csv` tiap wilayah, dan model `.joblib` yang sudah diekspor
`Dashboard/train_export.py` -- model yang sama persis dengan yang dipakai
dashboard, dan yang metriknya sudah diverifikasi terhadap `ringkasan_model.csv`.

### Dua hal yang harus dibawa saat membaca seluruh notebook ini

**Angka Diabetes yang tinggi bukan keberhasilan prediksi.** Label Diabetes pada
data ini praktis merupakan aturan `IMT > 25` yang dijalankan kuesioner
(notebook 2 Bagian 1, terbukti di kedua wilayah). Model yang "berhasil" hanya
menyusun ulang aturan itu.

**Setiap metrik terikat pada pita usianya.** Keempat penyakit ditanyakan pada
subpopulasi usia yang berbeda, jadi membandingkan Kanker Paru dengan TBC berarti
membandingkan dua populasi berbeda, bukan dua penyakit pada populasi yang sama.

In [ ]:
# ===================== IMPORTS =====================
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import joblib
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

AKAR = Path.cwd()
while not (AKAR / "Bersama").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "Bersama"))

import konfigurasi as cfg
import viz_style as vz

vz.apply_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

WILAYAH = ["Bantul", "Sleman"]          # wilayah dengan data mentah sendiri
SEMUA = WILAYAH + ["Gabungan"]
WARNA = vz.PALETTE["WILAYAH"]

MODEL_DIR = AKAR / "Gabungan" / "Dashboard" / "models"
SLUG_WILAYAH = {"Bantul": "bantul", "Sleman": "sleman", "Gabungan": "gabungan"}
SLUG = {"Diabetes Melitus": "diabetes", "Hipertensi": "hipertensi",
        "Kanker Paru": "kanker_paru", "TBC": "tbc"}
PENYAKIT = list(SLUG)

print("Wilayah yang dibandingkan:", ", ".join(SEMUA))

## 1. Tabel Performa Ketiga Kumpulan Model

`ringkasan_model.csv` tiap wilayah berisi keempat algoritma x keempat penyakit.
Ketiganya dibaca lalu ditumpuk menjadi satu tabel.

In [ ]:
# ===================== MUAT KETIGA RINGKASAN =====================
ringkasan = {}
for w in SEMUA:
    try:
        r = cfg.baca_ringkasan(w)
    except FileNotFoundError:
        print(f"({w} dilewati -- ringkasan_model.csv belum ada)")
        continue
    r.insert(0, "Wilayah", w)
    ringkasan[w] = r
    print(f"{w:<10} {len(r)} baris")

semua_r = pd.concat(ringkasan.values(), ignore_index=True)
ADA = list(ringkasan)


def model_terbaik(tab):
    """Definisi tunggal "model terbaik" -- dummy SELALU dikecualikan.

    Pada Hipertensi kelas positif adalah mayoritas, sehingga dummy mencetak F1
    lebih tinggi daripada model sungguhan mana pun. idxmax tanpa pengecualian
    ini akan memilih dummy dan menampilkannya sebagai "model terbaik".
    """
    kandidat = tab[tab["Model"] != "dummy"]
    return kandidat.loc[kandidat["F1-Score"].idxmax()]

In [ ]:
# ===================== MODEL TERBAIK PER WILAYAH PER PENYAKIT =====================
baris = []
for w in ADA:
    for p in PENYAKIT:
        sub = semua_r[(semua_r.Wilayah == w) & (semua_r.Dataset == p)]
        if sub.empty:
            continue
        r = model_terbaik(sub)
        baris.append({
            "Wilayah": w, "Penyakit": p, "Pita usia": r["Rentang usia"],
            "Prevalensi (%)": round(r["Prevalensi"] * 100, 2),
            "Model": r["Model"],
            **{k: round(r[k], 4) for k in
               ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]},
        })

terbaik = pd.DataFrame(baris)
display(terbaik.set_index(["Penyakit", "Wilayah"]).sort_index())

assert "dummy" not in set(terbaik["Model"]), "dummy terpilih -- ini bug"
print("Pemeriksaan lulus: dummy tidak pernah terpilih sebagai model terbaik.")

In [ ]:
# ===================== GRAFIK PERBANDINGAN METRIK =====================
METRIK = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
fig, axes = plt.subplots(2, 2, figsize=vz.FIG_2X2)

for ax, p in zip(axes.ravel(), PENYAKIT):
    sub = terbaik[terbaik.Penyakit == p].set_index("Wilayah")
    x = np.arange(len(METRIK))
    lebar = 0.8 / len(sub)
    for i, w in enumerate(sub.index):
        ax.bar(x + (i - (len(sub) - 1) / 2) * lebar, sub.loc[w, METRIK],
               width=lebar, color=WARNA[w], label=w)
    ax.set_xticks(x)
    ax.set_xticklabels(METRIK, rotation=20)
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color="grey", ls=":", lw=1)
    pita = sub["Pita usia"].iloc[0]
    ax.set_title(f"{p}  (pita usia {pita})", fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8)

fig.suptitle("Performa Model Terbaik per Wilayah -- garis titik = tebakan acak",
             fontsize=13)
plt.tight_layout()
plt.show()

print("Selisih maksimum antar wilayah, per penyakit per metrik:")
for p in PENYAKIT:
    sub = terbaik[terbaik.Penyakit == p][METRIK]
    print(f"   {p:<18} " + "  ".join(
        f"{m}={sub[m].max() - sub[m].min():.4f}" for m in METRIK))

Yang perlu dilihat di sini bukan wilayah mana yang menang, melainkan **seberapa
kecil selisihnya**. Model yang dilatih di kabupaten berbeda, pada data yang
dikumpulkan terpisah, sampai pada performa yang praktis sama untuk keempat
penyakit. Itu menunjukkan yang dipelajari model adalah pola yang sama, bukan
kekhasan satu kabupaten.

**Model Gabungan tidak otomatis lebih baik** meskipun dilatih pada data hampir
dua kali lipat. Ia juga dilatih pada populasi yang lebih heterogen, dan kedua
efek itu saling meniadakan. Tambahan data tidak membantu bila keterbatasannya
ada pada *variabel yang tersedia* -- dan pada TBC memang itu masalahnya.

In [ ]:
# ===================== DUMMY: DUA PELAJARAN YANG HARUS TETAP TERLIHAT =====================
dummy = semua_r[semua_r.Model == "dummy"][
    ["Wilayah", "Dataset", "Accuracy", "F1-Score", "ROC-AUC"]]
display(dummy.set_index(["Dataset", "Wilayah"]).round(4).sort_index())

print("1) TBC: dummy mencetak Accuracy tinggi dengan F1 = 0 -- ia menebak")
print("   tidak ada seorang pun yang berisiko, lalu benar hampir selalu.")
print()
print("2) Hipertensi: F1 dummy LEBIH TINGGI daripada model terbaik.")
for w in ADA:
    d = dummy[(dummy.Wilayah == w) & (dummy.Dataset == "Hipertensi")]
    t = terbaik[(terbaik.Wilayah == w) & (terbaik.Penyakit == "Hipertensi")]
    if d.empty or t.empty:
        continue
    print(f"   {w:<10} dummy F1 {d['F1-Score'].iloc[0]:.4f}  vs  "
          f"{t['Model'].iloc[0]} F1 {t['F1-Score'].iloc[0]:.4f}   "
          f"(ROC-AUC dummy {d['ROC-AUC'].iloc[0]:.3f} vs "
          f"{t['ROC-AUC'].iloc[0]:.3f})")
print()
print("   Sebabnya kelas positif Hipertensi adalah MAYORITAS, dan F1 mengabaikan")
print("   true negative. Karena itu dummy tidak pernah boleh dipilih sebagai")
print("   model terbaik, dan ROC-AUC yang membongkarnya: dummy selalu 0,500.")

## 2. Apakah Variabel Penting yang Sama?

Kepentingan fitur dibaca dari `metadata.json` tiap wilayah, yang ditulis
`train_export.py` -- jadi angkanya persis yang dipakai dashboard, bukan hasil
hitungan ulang yang bisa menyimpang.

Dua ukuran ditampilkan berdampingan, dan **keduanya benar untuk pertanyaannya
masing-masing**:

- **gain** menjumlahkan perbaikan pemisahan di setiap kali sebuah fitur dipakai
  memecah cabang. Fitur kategorik dengan banyak kategori mendapat banyak
  pemecahan kecil, dan semuanya ikut terhitung.
- **mean |SHAP|** mengukur rata-rata besar pengaruh sebuah fitur terhadap
  **prediksi peserta yang nyata**. Pemecahan kecil yang jarang kena hampir tidak
  berarti di sini.

> Jangan pernah menyebut salah satunya sebagai *jawaban*. Selalu sebutkan ukuran
> mana yang menghasilkan peringkat itu.

In [ ]:
# ===================== BACA KEPENTINGAN FITUR DARI metadata.json =====================
meta = {}
for w in SEMUA:
    q = MODEL_DIR / SLUG_WILAYAH[w] / "metadata.json"
    if q.exists():
        meta[w] = json.loads(q.read_text(encoding="utf-8"))
        print(f"{w:<10} metadata.json dibaca "
              f"({len(meta[w]['features'])} fitur)")
    else:
        print(f"({w} dilewati -- metadata.json belum ada)")

kep = []
for w, m in meta.items():
    for p, info in m["penyakit"].items():
        imp = info["importansi"]
        for ukuran in ("gain", "shap"):
            for fitur, nilai in imp[ukuran].items():
                kep.append({"Wilayah": w, "Penyakit": p, "Ukuran": ukuran,
                            "Fitur": fitur, "Nilai": nilai})

kep_df = pd.DataFrame(kep)

teratas = []
for w, m in meta.items():
    for p, info in m["penyakit"].items():
        imp = info["importansi"]
        teratas.append({
            "Wilayah": w, "Penyakit": p, "Model": info["model"],
            "Teratas gain": imp["teratas_gain"],
            "Teratas SHAP": imp["teratas_shap"],
            "Sepakat?": "ya" if imp["teratas_gain"] == imp["teratas_shap"] else "TIDAK",
        })

display(pd.DataFrame(teratas).set_index(["Penyakit", "Wilayah"]).sort_index())

In [ ]:
# ===================== GRAFIK KEPENTINGAN: GAIN vs SHAP, PER WILAYAH =====================
fig, axes = plt.subplots(len(PENYAKIT), 2, figsize=(15, 4 * len(PENYAKIT)))

for r, p in enumerate(PENYAKIT):
    for c, ukuran in enumerate(["gain", "shap"]):
        ax = axes[r, c]
        sub = kep_df[(kep_df.Penyakit == p) & (kep_df.Ukuran == ukuran)]
        piv = sub.pivot_table(index="Fitur", columns="Wilayah", values="Nilai")
        piv = piv.reindex(piv.mean(axis=1).sort_values().index)
        y = np.arange(len(piv))
        tinggi = 0.8 / len(piv.columns)
        for i, w in enumerate(piv.columns):
            ax.barh(y + (i - (len(piv.columns) - 1) / 2) * tinggi, piv[w],
                    height=tinggi, color=WARNA[w], label=w)
        ax.set_yticks(y)
        ax.set_yticklabels(piv.index, fontsize=9)
        ax.set_title(f"{p} -- {'gain' if ukuran == 'gain' else 'mean |SHAP|'}",
                     fontsize=11)
        ax.set_xlim(0, 1)
        ax.grid(axis="x", alpha=0.3)
        if r == 0 and c == 0:
            ax.legend(fontsize=8)

fig.suptitle("Kepentingan Fitur: gain (kiri) vs mean |SHAP| (kanan)", fontsize=13)
plt.tight_layout()
plt.show()

**TBC adalah kasus yang harus dibaca hati-hati, dan pola itu muncul di kedua
wilayah.** Menurut *gain*, `segmen` adalah fitur teratas; menurut *mean |SHAP|*,
`Usia`. Keduanya benar untuk pertanyaannya masing-masing, dan yang menentukan
mana yang relevan adalah apa yang ingin dikatakan:

- ingin menjelaskan **bagaimana model bekerja** -> gain
- ingin menjelaskan **apa yang menggerakkan prediksi seseorang** -> SHAP

SHAP juga sejalan dengan bukti tanpa model: tabel asosiasi di notebook 2 Bagian 4
menempatkan `Usia` di atas `segmen` untuk TBC, di kedua wilayah. Dua metode yang
saling bebas sepakat, dan gain-lah yang menyendiri.

Itu **memperhalus, bukan membatalkan**, temuan bahwa `segmen` penting untuk TBC.
Notebook 2 Bagian 5 menunjukkan pengaruh `segmen` tetap ada setelah usia
dikendalikan; yang mengecil adalah **ukurannya**, bukan keberadaannya.

Pada tiga penyakit lain gain dan SHAP sepakat, dan sepakat pula di kedua
wilayah: IMT untuk Diabetes dan Hipertensi, jenis kelamin untuk Kanker Paru.

## 3. Transfer Lintas Wilayah

Pertanyaan yang hanya bisa dijawab kalau ada dua kabupaten:

> **Apakah model yang dilatih di Bantul masih berguna di Sleman, dan
> sebaliknya?**

Caranya: ambil model Bantul yang sudah diekspor -- yang dilatih pada 80% data
Bantul -- lalu nilai ia pada **seluruh** data Sleman. Lalu kebalikannya. Tidak
ada kebocoran data: model Bantul tidak pernah melihat satu baris Sleman pun.

### Dua jebakan yang harus dihindari saat membaca hasilnya

**Prevalensi kedua wilayah berbeda.** Penurunan Precision bisa berarti model
gagal berpindah, atau sekadar berarti wilayah tujuan punya proporsi kasus yang
berbeda. Karena itu prevalensi masing-masing ikut dicetak, dan **ROC-AUC**
mendapat perhatian khusus: ia menilai *urutan* peserta menurut risiko, dan tidak
terpengaruh pergeseran prevalensi maupun letak ambang.

**Pembandingnya harus adil.** Skor "wilayah sendiri" diambil dari
`ringkasan_model.csv`, yaitu 20% data uji yang tidak pernah dilatihkan -- bukan
dari data yang sama dengan data latih.

In [ ]:
# ===================== TRANSFER: MUAT MODEL & NILAI SILANG =====================
def muat_model(wilayah, penyakit):
    return joblib.load(MODEL_DIR / SLUG_WILAYAH[wilayah] / f"model_{SLUG[penyakit]}.joblib")


def metrik(y, prob):
    pred = (prob >= 0.5).astype(int)
    return {
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1-Score": f1_score(y, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y, prob),
    }


FITUR6 = ["Jeniskelamin", "segmen", "Usia", "Tinggibadan", "Berat Badan",
          "IMT_recalculated"]

hasil_transfer = []
for asal in WILAYAH:
    for tujuan in WILAYAH:
        if asal == tujuan:
            continue
        for p in PENYAKIT:
            d = cfg.baca_dataset(tujuan, SLUG[p])
            target = [c for c in d.columns if c.startswith("Berisiko")][0]
            y = d[target].astype(int)
            prob = muat_model(asal, p).predict_proba(d[FITUR6])[:, 1]
            hasil_transfer.append({
                "Dilatih di": asal, "Diuji di": tujuan, "Penyakit": p,
                "n uji": len(d),
                "Prevalensi uji (%)": round(y.mean() * 100, 2),
                **{k: round(v, 4) for k, v in metrik(y, prob).items()},
            })
            print(f"  {asal} -> {tujuan}  {p:<18} n={len(d):>7,}")

transfer = pd.DataFrame(hasil_transfer)
display(transfer.set_index(["Penyakit", "Dilatih di"]).sort_index())

In [ ]:
# ===================== TRANSFER vs WILAYAH SENDIRI =====================
banding = []
for _, t in transfer.iterrows():
    sendiri = terbaik[(terbaik.Wilayah == t["Dilatih di"])
                      & (terbaik.Penyakit == t["Penyakit"])].iloc[0]
    tujuan_sendiri = terbaik[(terbaik.Wilayah == t["Diuji di"])
                             & (terbaik.Penyakit == t["Penyakit"])].iloc[0]
    banding.append({
        "Penyakit": t["Penyakit"],
        "Arah": f"{t['Dilatih di']} -> {t['Diuji di']}",
        "ROC-AUC di rumah": sendiri["ROC-AUC"],
        "ROC-AUC transfer": t["ROC-AUC"],
        "Selisih": round(t["ROC-AUC"] - sendiri["ROC-AUC"], 4),
        "ROC-AUC model lokal": tujuan_sendiri["ROC-AUC"],
        "Tertinggal dari lokal": round(t["ROC-AUC"] - tujuan_sendiri["ROC-AUC"], 4),
        "Prevalensi asal (%)": sendiri["Prevalensi (%)"],
        "Prevalensi tujuan (%)": t["Prevalensi uji (%)"],
    })

tabel_banding = pd.DataFrame(banding)
display(tabel_banding.set_index(["Penyakit", "Arah"]).sort_index())

print("Ringkasan ROC-AUC:")
print(f"   penurunan terbesar saat transfer : "
      f"{tabel_banding['Selisih'].min():+.4f}")
print(f"   penurunan rata-rata              : "
      f"{tabel_banding['Selisih'].mean():+.4f}")
print(f"   tertinggal terbesar dari model lokal wilayah tujuan: "
      f"{tabel_banding['Tertinggal dari lokal'].min():+.4f}")

In [ ]:
# ===================== GRAFIK TRANSFER =====================
fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

# Kiri: ROC-AUC -- tahan terhadap perbedaan prevalensi, jadi inilah ukuran
# utama untuk menilai apakah transfernya berhasil.
x = np.arange(len(PENYAKIT))
for i, arah in enumerate(tabel_banding["Arah"].unique()):
    sub = tabel_banding[tabel_banding.Arah == arah].set_index("Penyakit").reindex(PENYAKIT)
    axes[0].plot(x, sub["ROC-AUC transfer"], marker="o", lw=2,
                 label=f"transfer {arah}")
for i, w in enumerate(WILAYAH):
    sub = terbaik[terbaik.Wilayah == w].set_index("Penyakit").reindex(PENYAKIT)
    axes[0].plot(x, sub["ROC-AUC"], marker="s", ls="--", color=WARNA[w],
                 label=f"model lokal {w}")
axes[0].set_xticks(x)
axes[0].set_xticklabels(PENYAKIT, rotation=20)
axes[0].axhline(0.5, color="grey", ls=":", lw=1)
axes[0].set_ylabel("ROC-AUC")
axes[0].set_ylim(0.45, 1.0)
axes[0].set_title("ROC-AUC: transfer vs model lokal")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Kanan: F1 -- SENGAJA ditampilkan terpisah, karena ia bergerak mengikuti
# perbedaan prevalensi dan karenanya tidak boleh dipakai menyimpulkan
# keberhasilan transfer.
for arah in tabel_banding["Arah"].unique():
    sub = transfer[transfer.apply(
        lambda r: f"{r['Dilatih di']} -> {r['Diuji di']}" == arah, axis=1)]
    sub = sub.set_index("Penyakit").reindex(PENYAKIT)
    axes[1].plot(x, sub["F1-Score"], marker="o", lw=2, label=f"transfer {arah}")
for w in WILAYAH:
    sub = terbaik[terbaik.Wilayah == w].set_index("Penyakit").reindex(PENYAKIT)
    axes[1].plot(x, sub["F1-Score"], marker="s", ls="--", color=WARNA[w],
                 label=f"model lokal {w}")
axes[1].set_xticks(x)
axes[1].set_xticklabels(PENYAKIT, rotation=20)
axes[1].set_ylabel("F1-Score")
axes[1].set_title("F1 -- ikut bergeser bersama prevalensi, baca hati-hati")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Cara membaca hasil transfer

Bandingkan **ROC-AUC transfer** dengan **ROC-AUC model lokal wilayah tujuan**
(kolom `Tertinggal dari lokal`). Itu pertanyaan yang sesungguhnya: kalau Anda
sudah punya data kabupaten itu sendiri, seberapa banyak yang hilang bila memakai
model kabupaten sebelah?

- Selisih **mendekati nol** berarti modelnya benar-benar berpindah, dan
  kabupaten baru yang belum punya model sendiri dapat memakai model tetangganya.
- Selisih **besar dan negatif** berarti model mempelajari sesuatu yang khas
  wilayahnya, dan setiap kabupaten butuh modelnya sendiri.

Perhatikan kolom prevalensi di sebelahnya. F1 dan Precision **akan** bergeser
ketika prevalensi bergeser, bahkan bila kemampuan model mengurutkan peserta
sama sekali tidak berubah. Itulah sebabnya kesimpulan diambil dari ROC-AUC.

Konsisten dengan notebook 3: di sana fitur `wilayah` hampir tidak berpengaruh
setelah karakteristik lain diperhitungkan. Kalau kabupaten memang menambah
sedikit sekali informasi, model yang dilatih di satu kabupaten memang
seharusnya berpindah dengan baik. Kedua notebook menjawab pertanyaan yang sama
dari dua arah.

## 4. Ringkasan

In [ ]:
# ===================== RINGKASAN AKHIR =====================
print("=" * 78)
print("RINGKASAN PERBANDINGAN MODEL")
print("=" * 78)

print("\n1. Model terbaik per penyakit per wilayah")
display(terbaik.pivot_table(index="Penyakit", columns="Wilayah",
                            values="Model", aggfunc="first"))

print("2. F1-Score model terbaik")
display(terbaik.pivot_table(index="Penyakit", columns="Wilayah",
                            values="F1-Score").round(4))

print("3. ROC-AUC model terbaik")
display(terbaik.pivot_table(index="Penyakit", columns="Wilayah",
                            values="ROC-AUC").round(4))

print("4. Fitur teratas -- gain vs SHAP")
display(pd.DataFrame(teratas).pivot_table(
    index="Penyakit", columns="Wilayah",
    values=["Teratas gain", "Teratas SHAP"], aggfunc="first"))

print("5. Kehilangan ROC-AUC saat model dipakai di wilayah lain")
display(tabel_banding.pivot_table(index="Penyakit", columns="Arah",
                                  values="Tertinggal dari lokal").round(4))

### Yang layak masuk laporan

**Performa hampir tidak bergantung wilayah.** Model yang dilatih terpisah di dua
kabupaten sampai pada angka yang praktis sama, dan model gabungan tidak
mengungguli keduanya meski datanya hampir dua kali lipat.

**Peringkat kepentingan fitur juga sama di kedua wilayah** -- termasuk
ketidaksepakatan gain-vs-SHAP pada TBC, yang muncul lagi dengan pola yang persis
sama. Ketidaksepakatan itu ternyata sifat struktur datanya, bukan kebetulan satu
kabupaten.

**Transfer lintas wilayah dinilai dengan ROC-AUC**, karena F1 dan Precision ikut
bergeser mengikuti perbedaan prevalensi. Angka lengkapnya ada di Bagian 3.

### Batas yang tetap ada, dan tidak boleh hilang dari laporan

- **Diabetes bukan prediksi penyakit.** Modelnya menyusun ulang aturan
  `IMT > 25` milik kuesioner, di kedua wilayah.
- **Presisi Kanker Paru dan TBC rendah.** Sebagian besar catatan yang ditandai
  berisiko sebenarnya tidak berisiko -- konsekuensi yang disengaja dari memilih
  recall tinggi pada penyakit langka.
- **TBC terbatas oleh variabel yang tersedia, bukan oleh algoritma.** Tidak ada
  satu pun variabel yang berasosiasi kuat dengannya di kedua wilayah. Faktor
  risiko yang sesungguhnya -- status merokok, kepadatan hunian, gizi, kontak
  serumah -- tidak ada di data ini.
- **Seluruh angka berlaku per catatan skrining, bukan per orang**, dan hanya
  berlaku bagi orang yang **ikut skrining** -- bukan seluruh penduduk kedua
  kabupaten.